In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, models, Input
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

TensorFlow Version: 2.20.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training images:", x_train.shape)
print("Testing images:", x_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

 36724736/170498071 ━━━━━━━━━━━━━━━━━━━━ 21:30 10us/step

In [ ]:
class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

In [ ]:
train_counts = np.bincount(y_train.flatten(), minlength=10)
test_counts = np.bincount(y_test.flatten(), minlength=10)

dataset_summary = pd.DataFrame({
    'Class': class_names,
    'Training Images': train_counts,
    'Testing Images': test_counts
})

dataset_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(class_names, train_counts)
plt.title('Training Dataset Class Distribution')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [ ]:
basic_model = models.Sequential([
    Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

basic_model.summary()

In [ ]:
basic_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
basic_history = basic_model.fit(
    x_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
basic_loss, basic_accuracy = basic_model.evaluate(
    x_test,
    y_test,
    verbose=1
)

print("Basic CNN Accuracy:", round(basic_accuracy * 100, 2), "%")

In [ ]:
model = models.Sequential([
    Input(shape=(32, 32, 3)),

    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.30),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),

    layers.Dense(10, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=1
)

print("Enhanced CNN Accuracy:", round(test_accuracy * 100, 2), "%")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Enhanced CNN Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(['Training', 'Validation'])
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Enhanced CNN Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Training', 'Validation'])
plt.show()

In [ ]:
y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_test.flatten()

print(classification_report(
    y_true,
    y_pred_classes,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Enhanced CNN Confusion Matrix')
plt.show()

In [ ]:
print("Basic CNN Accuracy:", round(basic_accuracy * 100, 2), "%")
print("Enhanced CNN Accuracy:", round(test_accuracy * 100, 2), "%")
print("Improvement:", round((test_accuracy - basic_accuracy) * 100, 2), "%")

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)

base_model.trainable = False

transfer_model = models.Sequential([
    Input(shape=(32, 32, 3)), # Input matches original CIFAR-10 image size
    layers.Resizing(96, 96),  # Resizes images to 96x96
    layers.Rescaling(1./127.5, offset=-1), # Preprocesses for MobileNetV2
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

transfer_model.summary()

In [ ]:
transfer_model.summary()

In [ ]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
transfer_history = transfer_model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
transfer_loss, transfer_accuracy = transfer_model.evaluate(
    x_test,
    y_test,
    verbose=1
)

print("MobileNetV2 Accuracy:", round(transfer_accuracy * 100, 2), "%")

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

In [ ]:
transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
fine_tune_history = transfer_model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
fine_tune_loss, fine_tune_accuracy = transfer_model.evaluate(
    x_test,
    y_test,
    verbose=1
)

print(
    "Fine-Tuned MobileNetV2 Accuracy:",
    round(fine_tune_accuracy * 100, 2),
    "%"
)

In [ ]:
results = {
    'Basic CNN': basic_accuracy,
    'Enhanced CNN': test_accuracy,
    'MobileNetV2': transfer_accuracy,
    'Fine-Tuned MobileNetV2': fine_tune_accuracy
}

for name, accuracy in results.items():
    print(name + ":", round(accuracy * 100, 2), "%")

In [ ]:
model_names = list(results.keys())
accuracies = list(results.values())

plt.figure(figsize=(10, 5))
plt.bar(model_names, accuracies)
plt.title('Final Model Performance Comparison')
plt.xlabel('Model')
plt.ylabel('Test Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.show()

In [ ]:
best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]

print("Best Model:", best_model_name)
print("Best Accuracy:", round(best_accuracy * 100, 2), "%")

In [ ]:
final_predictions = transfer_model.predict(x_test)
final_pred_classes = np.argmax(final_predictions, axis=1)
final_true = y_test.flatten()

In [ ]:
print(classification_report(
    final_true,
    final_pred_classes,
    target_names=class_names
))

In [ ]:
final_cm = confusion_matrix(
    final_true,
    final_pred_classes
)

plt.figure(figsize=(10, 8))
sns.heatmap(
    final_cm,
    annot=True,
    fmt='d',
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Fine-Tuned MobileNetV2 Confusion Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])

    predicted = class_names[final_pred_classes[i]]
    actual = class_names[final_true[i]]

    plt.title(f"Pred: {predicted}\nActual: {actual}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
transfer_model.save('CIFAR10_Image_Classifier.keras')
print("Model saved successfully.")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array

image_path = list(uploaded.keys())[0]

img = load_img(image_path, target_size=(96, 96))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

prediction = transfer_model.predict(img_array, verbose=0)[0]

predicted_class = np.argmax(prediction)
confidence = prediction[predicted_class] * 100

print("Predicted Class:", class_names[predicted_class])
print("Confidence:", round(confidence, 2), "%")

In [ ]:
display_image = load_img(image_path)

plt.figure(figsize=(5, 5))
plt.imshow(display_image)
plt.title(
    f"Prediction: {class_names[predicted_class]}\n"
    f"Confidence: {confidence:.2f}%"
)
plt.axis('off')
plt.show()

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

In [ ]:
def predict_image(image):
    image = image.convert('RGB')
    image = image.resize((96, 96))

    image_array = np.array(image).astype('float32')
    image_array = np.expand_dims(image_array, axis=0)
    image_array = preprocess_input(image_array)

    prediction = transfer_model.predict(
        image_array,
        verbose=0
    )[0]

    return {
        class_names[i]: float(prediction[i])
        for i in range(10)
    }

In [ ]:
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="CIFAR-10 Image Classifier",
    description="Upload an image to classify it into one of the CIFAR-10 classes."
)

demo.launch()

In [ ]:
print("Basic CNN:", round(basic_accuracy * 100, 2), "%")
print("Enhanced CNN:", round(test_accuracy * 100, 2), "%")
print("MobileNetV2:", round(transfer_accuracy * 100, 2), "%")
print("Fine-Tuned MobileNetV2:", round(fine_tune_accuracy * 100, 2), "%")
print("Best Model:", best_model_name)
print("Best Accuracy:", round(best_accuracy * 100, 2), "%")